In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history
from data.constants import TRADING_DAYS_PER_YEAR

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

history.head()


## Model overview

**GARCH (Generalized Autoregressive Conditional Heteroskedasticity)**

Classical econometrics and basic models assume that the variance of errors is constant (homoscedastic). In real-world markets, this assumption is severely violated.

The model decomposes the return process into two equations: the mean equation and the conditional variance equation. We will review the univariate case of the model - **GARCH(1,1)**.

1. **Mean Equation**
    $$r_t = \mu + \epsilon_t, \quad \epsilon_t = \sigma_t z_t,$$
    where:
    * $r_t$ - the asset's log-return on day $t$;
    * $\mu$ - the constant mean (or an ARMA process);
    * $\epsilon_t$ - the innovation (error / market shock) on day $t$;
    * $\sigma_t$ - conditional volatility, which changes every day;
    * $z_t \sim \text{i.i.d.}$ $\mathcal{N}(0, 1)$ - standardized white noise (or a Student's $t$-distribution, with $\nu \rarr \infty$).
2. **Variance Equation**
    $$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
    Where the parameters have a clear economic interpretation:
    * $\omega$ (Omega, $\omega > 0$): The baseline level of variance (long-term baseline).
    * $\alpha$ (Alpha / ARCH parameter, $\alpha \ge 0$): Response to shocks. Indicates how strongly yesterday's unexpected market shock, $\epsilon_{t-1}^2$, affects today's volatility.
    * $\beta$ (Beta / GARCH parameter, $\beta \ge 0$): Process memory (persistence). Indicates how long volatility remains elevated after a shock has occurred.
3. **Key Properties and Parameter Constraints**
    * Stationarity Condition
        $$\alpha + \beta < 1,$$
        where  $(\alpha + \beta)$ - means _Persistance_
    * Long-Run (Unconditional) Volatility
        
        What mean level does volatility revert to over the long run?
        $$\sigma_{\text{long-term}}^2 = \frac{\omega}{1 - (\alpha + \beta)}$$
    * $k$-Step-Ahead Volatility Forecast ($t+k$)
        $$E[\sigma_{t+k}^2 \mid I_t] = \sigma_{\text{long-term}}^2 + (\alpha + \beta)^{k-1} (\sigma_{t+1}^2 - \sigma_{\text{long-term}}^2)$$

4. **How to fit GARCH? (Maximum Likelihood Estimation)**
    
    Since $\sigma_t^2$ - is not homoscedastic (it's hetheroscedastic) we cannot use Mean Squared Error (MSE) optimizer.
    The parameters are estimated numerically using the **Maximum Likelihood Estimation (MLE) method**:
    $$\ln L(\theta) = -\frac{1}{2} \sum_{t=1}^T \left( \ln(2\pi) + \ln(\sigma_t^2) + \frac{\epsilon_t^2}{\sigma_t^2} \right)$$

    In order to build the Likelihood function, we need to know the log-returns distribution. From the previous analysis we know that it is _Student's $t$-distribution_ with degrees of freedom - $\nu \approx 4.92$ (see _../market_analysis/data_distribution_analysis.ipynb_).

There are some modification for the GARCH model, like **EGARCH** - which models $\ln(\sigma_t^2)$, does not require $\omega, \alpha, \beta > 0$ constraints, and accounts for asymmetry. It is widely used for stocks and indexes with a strong _leverage-effect_ (the situation when the markets fall down faster and deeper that they grow).

Manual calculation (for **NVDA** only)

We are going to use Student's $t$-distribution for log-returns, so the Likelihood function can be written as:
$$\ln L(\mu, s, \nu) = T \ln\left[\frac{\Gamma\left(\frac{\nu+1}{2}\right)}{\Gamma\left(\frac{\nu}{2}\right)\sqrt{\pi (\nu - 2)} s}\right] - \frac{\nu+1}{2} \sum_{t=1}^T \ln\left(1 + \frac{(r_t - \mu)^2}{(\nu - 2) s^2}\right)$$

In [ ]:
from data.processors import log_returns
from scipy import optimize, special

log_ret = np.array(log_returns(history)['NVDA'] * 100)
n = log_ret.shape[0]
var_0 = log_ret.var()
mean_0 = log_ret.mean()

# Likelihood function (for Student's distr)
def garch_likelihood(params: np.ndarray) -> float:
    (w, alpha, beta, nu, mu) = params
    eps = 0.
    var = var_0
    dynamic_part = np.zeros(n)
    const_part = np.zeros(n)

    for i in range(0, n):
        var = var if i == 0 else w + alpha * eps + beta * var
        eps = (log_ret[i] - mu) ** 2.
        
        const_part[i] = special.gammaln((nu + 1) / 2.) - (special.gammaln(nu/2) + np.log(np.sqrt(np.pi * (nu - 2.) * var)))
        dynamic_part[i] = ((nu + 1) / 2.) * (np.log(1 + eps / ((nu - 2.) * var)))

    likelihood = (const_part - dynamic_part).sum()
    
    return -likelihood

x0 = [0.01, 0.1, 0.85, 4, mean_0] # w, alpha, beta, nu, mu
constraints = [
    {'type': 'ineq', 'fun': lambda x: -(x[1] + x[2] - 1)}
]
bounds = [(1e-6, None), (1e-6, 1), (1e-6, 1), (2.01, None), (None, None)]

# Fit the GARCH model
res = optimize.minimize(
    fun=lambda x: garch_likelihood(x),
    x0=x0,
    method='SLSQP',
    constraints=constraints,
    bounds=bounds,
)

(w, alpha, beta, nu, mu) = res.x if res.success else [0, 0, 0, 0, 0]

# Construct the last variance (for the last day, based on the optimal parameters)
eps = 0.
var_next_day = var_0
for i in range(0, n): # var_t
    var_next_day = w + alpha * eps + beta * var_next_day
    eps = (log_ret[i] - mu) ** 2

var_next_day = w + alpha * eps + beta * var_next_day # var_t+1
var_long = w / (1 - (alpha + beta ))

k = 5 # horizon
predict_var = np.array([var_long + (alpha + beta) ** (step - 1) * (var_next_day - var_long) for step in range(1, k + 1)])
cum_var = predict_var.sum()

daily_var = cum_var / k
predict_vol = np.sqrt(daily_var * TRADING_DAYS_PER_YEAR) / 100

expected_return = mu * TRADING_DAYS_PER_YEAR / 100

print(f"NVDA GARCH volatility for the {k} days horizon: {predict_vol:.3f}")
print("-------")
print(f"NVDA GARCH return for the {k} days horizon: {expected_return:.3f}")


Calculation for **NVDA** only using _arch_ package

In [ ]:
from data.processors import log_returns
from src.portfolio import garch

k = 5 # horizon
nvda_log_ret = np.array(log_returns(history)['NVDA'])

(nvda_next_period_returns, nvda_next_period_vol) = garch(log_returns=nvda_log_ret, t=k)

print(f"NVDA GARCH volatility for the {k} days horizon: {nvda_next_period_vol}")
print("-------")
print(f"NVDA GARCH returns for the {k} days horizon: {nvda_next_period_returns}")


Calculation for the whole portfolio using _arch_ package

In [ ]:
from data.processors import log_returns
from src.portfolio import garch

log_ret = np.array(log_returns(history))

k = 5 # horizon
(next_period_returns, next_period_vol) = garch(log_returns=log_ret, t=k)

print(f"GARCH volatility for the {k} days horizon: {next_period_vol}")
print("-------")
print(f"GARCH returns for the {k} days horizon: {next_period_returns}")


## Data Visualization

Find Sharpe ratio

In [ ]:
from src.portfolio import find_max_sharpe, get_risk_free_rate

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_garch_sharpe, garch_stocks_w) = find_max_sharpe(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='GARCH',
    returns_model='BLACK_LITTERMAN')

exact_garch_max_ret = max_garch_sharpe.tangency_return
exact_garch_max_vol = max_garch_sharpe.tangency_vol
exact_garch_max_sharpe = max_garch_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {exact_garch_max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_garch_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_garch_max_vol:.2%}")

print("Exact optimum stocks distribution:")
garch_stocks_w


Find Sharpe ratio (EGARCH)

In [ ]:
from src.portfolio import find_max_sharpe, get_risk_free_rate

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_egarch_sharpe, egarch_stocks_w) = find_max_sharpe(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='EGARCH',
    returns_model='BLACK_LITTERMAN')

exact_egarch_max_ret = max_egarch_sharpe.tangency_return
exact_egarch_max_vol = max_egarch_sharpe.tangency_vol
exact_egarch_max_sharpe = max_egarch_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {exact_egarch_max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_egarch_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_egarch_max_vol:.2%}")

print("Exact optimum stocks distribution:")
egarch_stocks_w


Find Sortino ratio

In [ ]:
from src.portfolio import find_max_sortino, get_risk_free_rate

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_garch_sortino, no_views_stocks_w) = find_max_sortino(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='GARCH',
    returns_model='BLACK_LITTERMAN')

max_garch_sortino_ret = max_garch_sortino.tangency_return
max_garch_sortino_vol = max_garch_sortino.tangency_vol
max_garch_sortino_sortino = max_garch_sortino.max_sortino

print(f"Exact Max Sortino Ratio: {max_garch_sortino_sortino:.4f}")
print(f"Exact Sortino Return: {max_garch_sortino_ret:.2%}")
print(f"Exact Sortino Volatility: {max_garch_sortino_vol:.2%}")

print("Exact optimum stocks distribution:")
no_views_stocks_w


Find Sortino EGARCH

In [ ]:
from src.portfolio import find_max_sortino, get_risk_free_rate

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_egarch_sortino, egarch_stocks_w) = find_max_sortino(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='EGARCH',
    returns_model='BLACK_LITTERMAN')

max_egarch_sortino_ret = max_egarch_sortino.tangency_return
max_egarch_sortino_vol = max_egarch_sortino.tangency_vol
max_egarch_sortino_sortino = max_egarch_sortino.max_sortino

print(f"Exact Max Sortino Ratio: {max_egarch_sortino_sortino:.4f}")
print(f"Exact Sortino Return: {max_egarch_sortino_ret:.2%}")
print(f"Exact Sortino Volatility: {max_egarch_sortino_vol:.2%}")

print("Exact optimum stocks distribution:")
egarch_stocks_w


Optimize portfolio for the Efficient Frontier visualization

In [ ]:
from src.portfolio import optimize_portfolio

optimum_df = optimize_portfolio(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='GARCH',
    returns_model='BLACK_LITTERMAN',
    prediction_period=5)
optimum_df


Visualize the data

In [ ]:
from data.processors import log_returns

fig, ax = plt.subplots(figsize=(12, 6))

log_ret = log_returns(history)
yr_cov = np.array(log_ret.cov() * TRADING_DAYS_PER_YEAR)  # type: ignore
num_assets = yr_cov.shape[0]
weights = np.ones(num_assets) / num_assets;

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()
expected_returns = log_ret.mean() * TRADING_DAYS_PER_YEAR

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(0.01, risk_free_rate + 0.01),
    arrowprops=dict(facecolor='black', width=0.5, headwidth=3, headlength=4, shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

# equal weights portfolio
x0 = np.ones(num_assets) / num_assets
def_x = float(np.sqrt(x0.T @ yr_cov @ x0))
def_y = np.dot(x0, expected_returns)
ax.scatter(
    def_x,
    def_y,
    color='darkred',
    alpha=0.7,
)
plt.annotate(
    'Equal weights portfolio',
    xy=(def_x, def_y),
    xytext=(def_x - 0.07, def_y + 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

# S&P 500 portfolio
sp500_df = download_tickers_history(start_date, end_date, ['^GSPC'])
sp500_log_ret = log_returns(sp500_df)

sp500_x = sp500_log_ret.std().iloc[0] * np.sqrt(TRADING_DAYS_PER_YEAR)
sp500_y = sp500_log_ret.mean().iloc[0] * TRADING_DAYS_PER_YEAR
ax.scatter(
    sp500_x,
    sp500_y,
    color='red',
    alpha=0.7,
)
plt.text(sp500_x + 0.001, sp500_y + 0.01, 'S&P 500 portfolio')

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the GARCH tangency portfolio point
plt.plot(exact_garch_max_vol, exact_garch_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio (GARCH)',
    xy=(exact_garch_max_vol, exact_garch_max_ret),
    xytext=(exact_garch_max_vol + 0.01, exact_garch_max_ret),
    arrowprops=dict(facecolor='black', width=1, headwidth=3, headlength=4, shrink=0.02)
)

# mark the GARCH max Sortino point
plt.plot(max_garch_sortino_vol, max_garch_sortino_ret, marker="*", markersize=8, markerfacecolor="lightgreen")
plt.annotate(
    'Max Sortino (GARCH)',
    xy=(max_garch_sortino_vol, max_garch_sortino_ret),
    xytext=(max_garch_sortino_vol - 0.05, max_garch_sortino_ret + 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

# mark the EGARCH tangency portfolio point
plt.plot(exact_egarch_max_vol, exact_egarch_max_ret, marker="*", markersize=8, markerfacecolor="orange")
plt.annotate(
    'Max Sharpe (EGARCH)',
    xy=(exact_egarch_max_vol, exact_egarch_max_ret),
    xytext=(exact_egarch_max_vol - 0.01, exact_egarch_max_ret - 0.03),
    arrowprops=dict(facecolor='black', width=1, headwidth=3, headlength=4, shrink=0.02)
)

# mark the EGARCH max Sortino point
plt.plot(max_egarch_sortino_vol, max_egarch_sortino_ret, marker="*", markersize=8, markerfacecolor="darkgreen")
plt.annotate(
    'Max Sortino (EGARCH)',
    xy=(max_egarch_sortino_vol, max_egarch_sortino_ret),
    xytext=(max_egarch_sortino_vol - 0.05, max_egarch_sortino_ret + 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

plt.title('Efficient Frontier')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
